In [ ]:
!pip3 install textattack[tensorflow]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 31.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Sentiment analysis

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# Test analysis of the original sentence
pipe = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
input_text = "The movie was a heartfelt and inspiring narrative."
pipe(input_text)

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9998834133148193}]

In [ ]:
import textattack
from textattack.attack_recipes.deepwordbug_gao_2018 import DeepWordBugGao2018


tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

attack = DeepWordBugGao2018.build(model_wrapper)

label = 1 # Positive
attack_result = attack.attack(input_text, label)
print(attack_result)

textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


1 (100%) --> 0 (96%)

The movie was a heartfelt and inspiring narrative.

The movie was a hearftelt and inspirBng narrative.


A single letter substitution in the word “inspiring” changed the classification of the whole sentence.

# Hate Speech

In [ ]:
from textattack.transformations import WordInsertion
from textattack.goal_functions import ClassificationGoalFunction
import string



class WhitspacePunctuationInsertion(WordInsertion):
  def __init__(self, limited, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.limited = limited

  # Limited: only add a small set of puntation and whithespaces
  def _get_new_words(self, current_text, index):
        """Returns a set of new words we can insert at position `index` of `current_text`
        Args:
            current_text (AttackedText): Current text to modify.
            index (int): Position in which to insert a new word
        Returns:
            list[str]: List of new words to insert.
        """
        if self.limited:
          words = [' ', ',']
        else:
          words = [i for i in string.punctuation] + [i for i in string.whitespace]
        return words



In [ ]:
pipe = pipeline("text-classification", model="facebook/roberta-hate-speech-dynabench-r4-target")

config.json:   0%|          | 0.00/816 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# The text is similar to one tweet from: https://github.com/albanyan/counterhate_reply/blob/main/Data/dataset.csv

In [ ]:
input_text = """Yea,we could only stay at home now...
Fuck you China!🤬

BoycottChina"""

In [ ]:
pipe(input_text)

[{'label': 'hate', 'score': 0.932158887386322}]

In [ ]:
import textattack
from transformers import AutoTokenizer, AutoModelForSequenceClassification


tokenizer = AutoTokenizer.from_pretrained("facebook/roberta-hate-speech-dynabench-r4-target")
model = AutoModelForSequenceClassification.from_pretrained("facebook/roberta-hate-speech-dynabench-r4-target")
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)



from textattack.search_methods import GreedySearch

# Untargeted: attempts to minimize the score of the correct label until it is no longer the predicted label
goal_function = textattack.goal_functions.UntargetedClassification(model_wrapper)

# No constrains needed, since I am only insering whitespaces and punctiation
constraints = [
]


transformation = WhitspacePunctuationInsertion(limited=True)

# greedily chooses from a list of possible perturbations
search_method = GreedySearch()

# Construct the actual attack
attack = textattack.Attack(goal_function, constraints, transformation, search_method)


label = 1 # Hate
attack_result = attack.attack(input_text, label)
print(attack_result)

textattack: Unknown if model of class <class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


1 (93%) --> 0 (60%)

Yea,we could only stay at home now...
Fuck you China!🤬
 
BoycottChina

    Yea,we could only stay at home now...
Fuck   you     China!🤬
 
BoycottChina


The main differences is that textattack added a few whitespaces at the beginning of the sentences and then between the words "Fuck you China", which seem to have confused the model